In [ ]:
#CELULA 1: Imports, Variaveis de Ambiente, GCP Client e Configuracoes Globais

import os
import io
import zipfile
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client GCP
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais Dinâmicos com Data Piso
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"
BUCKET_SILVER = "cvm-lakehouse-silver-dev"

HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)

DATA_PISO_BACKFILL = datetime.strptime("2026-01-01", "%Y-%m-%d").date()

bucket_bronze = client.bucket(BUCKET_BRONZE)
bucket_silver = client.bucket(BUCKET_SILVER)

pd.set_option("display.max_columns", None)

print(f"✔ Client GCP inicializado para CAMADA SILVER!")
print(f"✔ Source Bronze: gs://{BUCKET_BRONZE}")
print(f"✔ Target Silver: gs://{BUCKET_SILVER}")
print(f"✔ Data Limite D-1: {D1_LIMITE.strftime('%d/%m/%Y')}")

In [ ]:
# =============================================================================
# PIPELINE DEFINITIVO: SCREENING DE FUNDOS DE CRÉDITO PRIVADO (CVM 175)
# =============================================================================
# -----------------------------------------------------------------------------
# 0. PREPARAÇÃO E DEDUPLICAÇÃO DA CARTEIRA MAIS RECENTE (D-90)
# -----------------------------------------------------------------------------
def preparar_carteira_recente(df_ativos):
    """Isola apenas a carteira (mês/competência) mais recente reportada por cada CNPJ."""
    print("⏳ Preparando base de ativos (isolando a carteira mais recente por fundo)...")
    
    # Trata valores numéricos
    df_ativos['VL_MERC_POS_FINAL'] = pd.to_numeric(
        df_ativos['VL_MERC_POS_FINAL'].astype(str).str.replace(',', '.'), errors='coerce'
    ).fillna(0)
    
    # Isola a data máxima por fundo
    col_data = 'DT_COMPTC' if 'DT_COMPTC' in df_ativos.columns else 'DT_POSICAO'
    max_datas = df_ativos.groupby('CNPJ_CLEAN')[col_data].max().reset_index()
    
    # Retorna o dataframe apenas com a foto do último mês
    return df_ativos.merge(max_datas, on=['CNPJ_CLEAN', col_data], how='inner')


# -----------------------------------------------------------------------------
# FUNÇÕES DE AUDITORIA E MENSAGERIA
# -----------------------------------------------------------------------------
def auditar_etapa(nome_etapa, df_in, df_out, cnpjs_quantum=None):
    """Gera o relatório de volumetria de cada etapa do funil."""
    total_in = len(df_in)
    total_out = len(df_out)
    removidos = total_in - total_out
    aum_out = df_out['VL_TOTAL_CARTEIRA'].sum() if 'VL_TOTAL_CARTEIRA' in df_out.columns else 0
    
    print(f"\n{'='*80}\n📊 {nome_etapa.upper()}\n{'='*80}")
    print(f"• Entrantes : {total_in:,} fundos")
    print(f"• Aprovados : {total_out:,} ({(total_out/total_in)*100:.1f}%) | AUM Retido: R$ {aum_out/1e9:,.2f} Bi")
    print(f"• Filtrados : {removidos:,} ({(removidos/total_in)*100:.1f}%)")
    
    if cnpjs_quantum:
        matches = set(df_out['CNPJ_CLEAN']).intersection(cnpjs_quantum)
        pct_aderencia = (len(matches)/len(cnpjs_quantum))*100 if len(cnpjs_quantum) > 0 else 0
        print(f"🎯 Quantum Retidos: {len(matches):,} de {len(cnpjs_quantum):,} ({pct_aderencia:.1f}%)")
    print("-" * 80)


# -----------------------------------------------------------------------------
# FUNÇÕES DO FUNIL DE FILTRAGEM (ETAPAS 1 A 6)
# -----------------------------------------------------------------------------
def aplicar_etapa_1(df_carteira):
    """Etapa 1: Mínimo de 10% em Crédito Direto (inclui FIDC/CCB) OU ser FIC (Cotas >= 50%)"""
    PAPEIS_CREDITO = [
        'Debênture simples', 'Letra Financeira', 'CDB/ RDB', 'CRA', 
        'Certificado de recebíveis imobiliários', 'Nota Promissória/ Commercial Paper/ Export Note',
        'DPGE', 'Debênture Permutável', 'CDB Vinculado', 'Debênture conversível', 'FIDC', 'CCB'
    ]
    PAPEIS_COTAS = ['Fundo de Investimento e de Cotas', 'FI Participações']

    aloc = df_carteira.groupby('CNPJ_CLEAN').agg(
        VL_CREDITO=('VL_MERC_POS_FINAL', lambda x: x[df_carteira.loc[x.index, 'TP_ATIVO'].isin(PAPEIS_CREDITO)].sum()),
        VL_COTAS=('VL_MERC_POS_FINAL', lambda x: x[df_carteira.loc[x.index, 'TP_ATIVO'].isin(PAPEIS_COTAS)].sum()),
        VL_TOTAL_CARTEIRA=('VL_MERC_POS_FINAL', 'sum')
    ).reset_index()

    df_base = aloc[aloc['VL_TOTAL_CARTEIRA'] > 0].copy()
    df_base['PCT_CREDITO'] = df_base['VL_CREDITO'] / df_base['VL_TOTAL_CARTEIRA']
    df_base['PCT_COTAS'] = df_base['VL_COTAS'] / df_base['VL_TOTAL_CARTEIRA']
    df_base['TP_ESTRUTURA'] = np.where(df_base['PCT_COTAS'] >= 0.50, 'FIC', 'MASTER/DIRETO')

    cond_e1 = (df_base['PCT_CREDITO'] >= 0.10) | (df_base['PCT_COTAS'] >= 0.50)
    return df_base[cond_e1].copy()

def aplicar_etapa_2(df_e1, df_cad):
    """Etapa 2: Exclusão de Fundos com mandato de Exterior / IE"""
    df_cad_limpo = df_cad.copy()
    if 'CNPJ_FUNDO_CLEAN' in df_cad_limpo.columns and 'CNPJ_CLEAN' not in df_cad_limpo.columns:
        df_cad_limpo['CNPJ_CLEAN'] = df_cad_limpo['CNPJ_FUNDO_CLEAN'].astype(str).str.replace(r'\D', '', regex=True)

    cols_cad = [c for c in ['DENOM_SOCIAL', 'RAZAO_SOCIAL', 'INVEST_EXTER'] if c in df_cad_limpo.columns]
    df_e2_in = df_e1.merge(df_cad_limpo[['CNPJ_CLEAN'] + cols_cad].drop_duplicates('CNPJ_CLEAN'), on='CNPJ_CLEAN', how='left')

    col_nome = [c for c in ['DENOM_SOCIAL', 'RAZAO_SOCIAL'] if c in df_e2_in.columns][0]
    REGEX_EXT = r'EXTERIOR|INVESTIMENTO NO EXTERIOR|OFFSHORE|\bIE\b|-IE|\(IE\)|FIEX'
    
    cond_nome = df_e2_in[col_nome].astype(str).str.upper().str.contains(REGEX_EXT, na=False, regex=True)
    cond_flag = (df_e2_in['INVEST_EXTER'].astype(str).str.upper() == 'S') if 'INVEST_EXTER' in df_e2_in.columns else False
    
    return df_e2_in[~(cond_nome | cond_flag)].copy()

def aplicar_etapa_3(df_e2, df_carteira):
    """Etapa 3: Teto de 30% em FIDC (exclui fundos muito concentrados em direitos creditórios)"""
    PAPEIS_FIDC = ['FIDC', 'FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS']
    
    carteira_e2 = df_carteira[df_carteira['CNPJ_CLEAN'].isin(df_e2['CNPJ_CLEAN'])]
    aloc_fidc = carteira_e2[carteira_e2['TP_ATIVO'].isin(PAPEIS_FIDC)].groupby('CNPJ_CLEAN').agg(
        VL_FIDC=('VL_MERC_POS_FINAL', 'sum')
    ).reset_index()

    df_e3_in = df_e2.merge(aloc_fidc, on='CNPJ_CLEAN', how='left')
    df_e3_in['VL_FIDC'] = df_e3_in['VL_FIDC'].fillna(0)
    df_e3_in['PCT_FIDC'] = df_e3_in['VL_FIDC'] / df_e3_in['VL_TOTAL_CARTEIRA']
    
    return df_e3_in[df_e3_in['PCT_FIDC'] <= 0.30].copy()

def aplicar_etapa_4(df_e3, df_carteira):
    """Etapa 4: Exclusão de Fundos Soberanos / Caixa (Teto de 95% em Títulos Públicos)"""
    PAPEIS_SOB = ['Título público federal', 'Operação compromissada']
    
    carteira_e3 = df_carteira[df_carteira['CNPJ_CLEAN'].isin(df_e3['CNPJ_CLEAN'])]
    aloc_sob = carteira_e3[carteira_e3['TP_ATIVO'].astype(str).str.upper().isin([p.upper() for p in PAPEIS_SOB])].groupby('CNPJ_CLEAN').agg(
        VL_SOBERANO=('VL_MERC_POS_FINAL', 'sum')
    ).reset_index()

    df_e4_in = df_e3.merge(aloc_sob, on='CNPJ_CLEAN', how='left')
    df_e4_in['VL_SOBERANO'] = df_e4_in['VL_SOBERANO'].fillna(0)
    df_e4_in['PCT_SOBERANO'] = df_e4_in['VL_SOBERANO'] / df_e4_in['VL_TOTAL_CARTEIRA']

    col_nome = [c for c in ['DENOM_SOCIAL', 'RAZAO_SOCIAL'] if c in df_e4_in.columns][0]
    REGEX_SOB = r'SOBERANO|DI SIMPLES|TESOURO SELIC|REFERENCIADO DI'
    
    cond_pct = df_e4_in['PCT_SOBERANO'] >= 0.95
    cond_nome = df_e4_in[col_nome].astype(str).str.upper().str.contains(REGEX_SOB, na=False, regex=True)
    
    return df_e4_in[~(cond_pct | cond_nome)].copy()

def aplicar_etapa_5(df_e4, df_carteira):
    """Etapa 5: Exclusão de Multimercados Híbridos (Teto de 3% em Ações) - Universo Regulatório Final"""
    PAPEIS_ACOES = ['Ação ordinária', 'Ação preferencial', 'UNITS', 'CERTIFICADOS DE DEPÓSITO DE AÇÕES', 'Ações', 'AÇÕES']
    
    carteira_e4 = df_carteira[df_carteira['CNPJ_CLEAN'].isin(df_e4['CNPJ_CLEAN'])]
    aloc_acoes = carteira_e4[carteira_e4['TP_ATIVO'].astype(str).str.upper().isin([p.upper() for p in PAPEIS_ACOES])].groupby('CNPJ_CLEAN').agg(
        VL_ACOES=('VL_MERC_POS_FINAL', 'sum')
    ).reset_index()

    df_e5_in = df_e4.merge(aloc_acoes, on='CNPJ_CLEAN', how='left')
    df_e5_in['VL_ACOES'] = df_e5_in['VL_ACOES'].fillna(0)
    df_e5_in['PCT_ACOES'] = df_e5_in['VL_ACOES'] / df_e5_in['VL_TOTAL_CARTEIRA']
    
    return df_e5_in[df_e5_in['PCT_ACOES'] <= 0.03].copy()

def aplicar_etapa_6_comercial(df_e5, df_cad):
    """Etapa 6 (Bônus): Filtro Comercial - Remove Fundos Exclusivos para isolar a Prateleira"""
    df_cad_limpo = df_cad.copy()
    if 'CNPJ_FUNDO_CLEAN' in df_cad_limpo.columns and 'CNPJ_CLEAN' not in df_cad_limpo.columns:
        df_cad_limpo['CNPJ_CLEAN'] = df_cad_limpo['CNPJ_FUNDO_CLEAN'].astype(str).str.replace(r'\D', '', regex=True)

    cols_comerciais = [c for c in ['FUNDO_EXCLUSIVO', 'SIT'] if c in df_cad_limpo.columns]
    df_e6_in = df_e5.merge(df_cad_limpo[['CNPJ_CLEAN'] + cols_comerciais].drop_duplicates('CNPJ_CLEAN'), on='CNPJ_CLEAN', how='left')
    
    if 'FUNDO_EXCLUSIVO' in df_e6_in.columns:
        cond_aberto = df_e6_in['FUNDO_EXCLUSIVO'].astype(str).str.upper() == 'N'
        return df_e6_in[cond_aberto].copy()
    return df_e6_in.copy()


# -----------------------------------------------------------------------------
# MAIN EXECUTOR
# -----------------------------------------------------------------------------
def rodar_pipeline_completo(df_ativos_bruto, df_cad, df_quantum=None):
    print("🚀 INICIANDO PIPELINE DE SCREENING DE CRÉDITO PRIVADO...\n")
    
    cnpjs_quantum = set(df_quantum['CNPJ_CLEAN']) if df_quantum is not None else None

    # Step 0: Deduplicação
    df_carteira = preparar_carteira_recente(df_ativos_bruto)
    
    # Execução das Etapas
    df_e1 = aplicar_etapa_1(df_carteira)
    auditar_etapa("Etapa 1: Filtro Base (Crédito & FICs)", df_carteira.drop_duplicates('CNPJ_CLEAN'), df_e1, cnpjs_quantum)

    df_e2 = aplicar_etapa_2(df_e1, df_cad)
    auditar_etapa("Etapa 2: Exclusão Exterior / IE", df_e1, df_e2, cnpjs_quantum)

    df_e3 = aplicar_etapa_3(df_e2, df_carteira)
    auditar_etapa("Etapa 3: Teto 30% FIDC", df_e2, df_e3, cnpjs_quantum)

    df_e4 = aplicar_etapa_4(df_e3, df_carteira)
    auditar_etapa("Etapa 4: Exclusão Soberanos/Caixa (< 95%)", df_e3, df_e4, cnpjs_quantum)

    df_e5 = aplicar_etapa_5(df_e4, df_carteira)
    auditar_etapa("Etapa 5: Teto 3% Ações (Universo Regulatório Final)", df_e4, df_e5, cnpjs_quantum)
    
    df_e6 = aplicar_etapa_6_comercial(df_e5, df_cad)
    auditar_etapa("Etapa 6: Filtro Comercial (Sem Exclusivos - Prateleira Real)", df_e5, df_e6, cnpjs_quantum)
    
    print("🏆 PIPELINE CONCLUÍDO COM SUCESSO!")
    return {
        'carteira_recente': df_carteira,
        'universo_regulatorio': df_e5,
        'prateleira_comercial': df_e6
    }

# =============================================================================
# INICIALIZAÇÃO AUTOMÁTICA
# =============================================================================
try:
    # Identifica a base cadastral disponível
    cadastro_ativo = df_cad_url_completo if 'df_cad_url_completo' in globals() else (df_cad_oficial if 'df_cad_oficial' in globals() else df_cad)
    quantum_ativo = df_quantum_oficial if 'df_quantum_oficial' in globals() else None
    ativos_bruto = df_ativos_detalhado if 'df_ativos_detalhado' in globals() else df_carteira_recente

    # Executa a pipeline e armazena os resultados num dicionário
    resultados_pipeline = rodar_pipeline_completo(ativos_bruto, cadastro_ativo, quantum_ativo)
    
    # Extrai os dataframes finais para uso direto
    df_universo_regulatorio = resultados_pipeline['universo_regulatorio'] # Os ~12.300 fundos
    df_prateleira_comercial = resultados_pipeline['prateleira_comercial'] # Os fundos abertos ao público
    
except NameError as e:
    print(f"❌ ERRO: Faltam os DataFrames de entrada na memória. Certifique-se de carregar os dados brutos primeiro. Detalhe: {e}")

In [ ]:
# =============================================================================
# DEMONSTRAÇÃO PRÁTICA: O ROTEIRO PARA OS GESTORES EM DADOS REAIS (BLINDADO)
# =============================================================================

import pandas as pd

print("=== EVIDÊNCIAS DOS 3 PILARES DO ROTEIRO (UNIVERSO VS. COMERCIAL) ===\n")

if 'cadastro_ativo' in globals() and 'df_universo_regulatorio' in globals():
    df_cad_demo = cadastro_ativo.copy()
    df_demo = df_universo_regulatorio.copy()
    
    # 1. Padroniza CNPJ no cadastro
    col_cnpj = 'CNPJ_CLEAN' if 'CNPJ_CLEAN' in df_cad_demo.columns else 'CNPJ_FUNDO_CLEAN'
    if col_cnpj != 'CNPJ_CLEAN':
        df_cad_demo['CNPJ_CLEAN'] = df_cad_demo[col_cnpj].astype(str).str.replace(r'\D', '', regex=True)
        
    # 2. Garante que FUNDO_EXCLUSIVO venha do cadastro sem duplicar colunas
    if 'FUNDO_EXCLUSIVO' in df_cad_demo.columns and 'FUNDO_EXCLUSIVO' not in df_demo.columns:
        df_demo = df_demo.merge(
            df_cad_demo[['CNPJ_CLEAN', 'FUNDO_EXCLUSIVO']].drop_duplicates('CNPJ_CLEAN'), 
            on='CNPJ_CLEAN', 
            how='left'
        )
        
    # 3. Identifica a coluna de nome que JÁ ESTÁ no df_demo
    nomes_possiveis = ['DENOM_SOCIAL', 'RAZAO_SOCIAL', 'NM_FUNDO']
    col_nome_ativa = [c for c in nomes_possiveis if c in df_demo.columns]
    
    # Se por acaso não estiver, trazemos do cadastro
    if not col_nome_ativa:
        col_cad = [c for c in nomes_possiveis if c in df_cad_demo.columns]
        if col_cad:
            df_demo = df_demo.merge(df_cad_demo[['CNPJ_CLEAN', col_cad[0]]].drop_duplicates('CNPJ_CLEAN'), on='CNPJ_CLEAN', how='left')
            nome_col = col_cad[0]
        else:
            nome_col = 'CNPJ_CLEAN'
    else:
        nome_col = col_nome_ativa[0]

    # -------------------------------------------------------------------------
    # TÓPICO 1: A Ilusão da Estrutura Master/Feeder (Multiplicação de CNPJs)
    # -------------------------------------------------------------------------
    print("1. A ILUSÃO DA ESTRUTURA MASTER/FEEDER (MULTIPLICAÇÃO DE CNPJS)")
    print("Muitos fundos na CVM são apenas 'espelhos' (FICs) vendendo a mesma estratégia.")
    
    if nome_col != 'CNPJ_CLEAN':
        df_demo['PRIMEIRO_NOME'] = df_demo[nome_col].astype(str).str.split().str[:2].str.join(' ')
        familias = df_demo[df_demo['TP_ESTRUTURA'] == 'FIC']['PRIMEIRO_NOME'].value_counts()
        
        if not familias.empty:
            nome_familia_exemplo = familias.index[0]
            qtd_espelhos = familias.iloc[0]
            print(f"-> A marca/gestora '{nome_familia_exemplo}' possui {qtd_espelhos} FICs (CNPJs diferentes) na CVM.")
            print(f"Para o Quantum, isso pode ser 1 ou 2 produtos. Para a CVM, são {qtd_espelhos} fundos.\n")
    else:
        print("-> [Coluna de nome não encontrada para agrupar as famílias de FICs.]\n")

    # -------------------------------------------------------------------------
    # TÓPICO 2: Fundos Exclusivos e de Tesouraria (O Mercado Oculto)
    # -------------------------------------------------------------------------
    print("2. FUNDOS EXCLUSIVOS E DE TESOURARIA (O MERCADO OCULTO)")
    print("Passaram no filtro de Crédito Privado, mas são veículos fechados (1 cotista).")
    
    if 'FUNDO_EXCLUSIVO' in df_demo.columns:
        df_exclusivos = df_demo[df_demo['FUNDO_EXCLUSIVO'].astype(str).str.upper() == 'S']
        aum_exclusivos = df_exclusivos['VL_TOTAL_CARTEIRA'].sum()
        
        print(f"-> O pipeline encontrou {len(df_exclusivos):,} Fundos Exclusivos retendo R$ {aum_exclusivos/1e9:,.2f} Bi.")
        print("Exemplos de veículos fechados na base:")
        for nome in df_exclusivos[nome_col].dropna().head(3):
            print(f"   - {nome}")
    else:
        print("-> [A coluna 'FUNDO_EXCLUSIVO' não está presente no cadastro para gerar os exemplos.]")
    print("\n")

    # -------------------------------------------------------------------------
    # TÓPICO 3: O Filtro de Viabilidade Comercial (Tamanho e Liquidez)
    # -------------------------------------------------------------------------
    print("3. O FILTRO DE VIABILIDADE COMERCIAL (TAMANHO MÍNIMO)")
    print("Fundos abertos ao público, mas pequenos demais ou novos demais para o Quantum mostrar.")
    
    if 'FUNDO_EXCLUSIVO' in df_demo.columns:
        df_abertos = df_demo[df_demo['FUNDO_EXCLUSIVO'].astype(str).str.upper() == 'N']
    else:
        df_abertos = df_demo
        
    df_pequenos = df_abertos[df_abertos['VL_TOTAL_CARTEIRA'] < 5_000_000]
    
    print(f"-> Existem {len(df_pequenos):,} fundos NÃO exclusivos, mas com AUM menor que R$ 5 Milhões.")
    print("As plataformas cortam esses fundos da tela por falta de porte/liquidez. Exemplos:")
    for _, row in df_pequenos.head(3).iterrows():
        print(f"   - {row[nome_col]} | AUM: R$ {row['VL_TOTAL_CARTEIRA']/1e6:,.2f} Milhões")
        
    print("\n" + "=" * 80)
else:
    print("❌ ERRO: Execute o pipeline completo primeiro para gerar os DataFrames.")

In [ ]:
import pandas as pd
import os

print("=== AMOSTRAGEM PARA DIAGNÓSTICO DE MERGE ===")

# 1. Isola as colunas de interesse
df_amostra_base = df_universo_regulatorio[['CNPJ_CLEAN']].head(15).copy()

col_cnpj_cad = 'CNPJ_CLEAN' if 'CNPJ_CLEAN' in cadastro_ativo.columns else 'CNPJ_FUNDO_CLEAN'
cols_cad_amostra = [col_cnpj_cad] + [c for c in ['FUNDO_EXCLUSIVO', 'SIT', 'SIT_FUNDO'] if c in cadastro_ativo.columns]
df_amostra_cad = cadastro_ativo[cols_cad_amostra].dropna(subset=[col_cnpj_cad]).head(15).copy()

# 2. Imprime na tela para visualização rápida
print("\n🔍 COMO O CNPJ ESTÁ NO UNIVERSO REGULATÓRIO (df_universo_regulatorio):")
print(df_amostra_base['CNPJ_CLEAN'].tolist())

print("\n🔍 COMO O CNPJ ESTÁ NO CADASTRO (cadastro_ativo):")
print(df_amostra_cad[col_cnpj_cad].tolist())

# 3. Salva os arquivos CSV para você anexar
caminho_base = "amostra_universo.csv"
caminho_cad = "amostra_cadastro.csv"

df_amostra_base.to_csv(caminho_base, index=False, sep=';')
df_amostra_cad.to_csv(caminho_cad, index=False, sep=';')

print(f"\n✅ Arquivos gerados no diretório: {os.getcwd()}")
print(f"Pode anexar os arquivos '{caminho_base}' e '{caminho_cad}' aqui no chat!")

In [ ]:
# CÉLULA: DESCOBRINDO OS VALORES REAIS DO CADASTRO
import pandas as pd

df_cad_teste = cadastro_ativo.copy()

print("=== VALORES ÚNICOS NA COLUNA FUNDO_EXCLUSIVO ===")
if 'FUNDO_EXCLUSIVO' in df_cad_teste.columns:
    print(df_cad_teste['FUNDO_EXCLUSIVO'].dropna().unique())
else:
    print("Coluna não existe.")

print("\n=== VALORES ÚNICOS NA COLUNA SIT ===")
col_sit = [c for c in ['SIT', 'SIT_FUNDO'] if c in df_cad_teste.columns]
if col_sit:
    print(df_cad_teste[col_sit[0]].dropna().unique()[:10]) # Mostra os 10 primeiros
else:
    print("Coluna não existe.")